In [31]:
models = ['DLinear', 'LightTS', 'MTSMixer', 
          'TimesNet', 'ModernTCN', 'TimeMixerPP', 
          'FEDformer', 'ETSformer', 'Crossformer', 'PatchTST', 'GPT4TS', 'iTransformer',
          'InterpretGN', 'TSCMamba',
          'MambaSL', 
]
datasets = ['EigenWorms', 'InsectWingbeat', 'DuckDuckGeese']

df = []

for model in models:
    for dataset in datasets:
        total_time, model_param = 0, 0

        test_out_file = f'_test_results/{model}_All_UEA30.out'
        ckpt_path = ''
        with open(test_out_file, 'r') as f:
            lines = f.readlines()
            for i, line in enumerate(lines):
                if line.startswith(f'>>>>>>>testing : classification_CLS_{dataset}_'):
                    ckpt_path = line.replace('>>>>>>>testing : ', '').replace('<', '').strip()
                    break

        full_out_file = f'../03-full_results/{model}/{model}_CLS_{dataset}.out' if model != 'MambaSL' \
                        else f'../03-full_results/{model}/_proposed/{model}_CLS_{dataset}_proposed.out'

        print(ckpt_path)
        with open(full_out_file, 'r') as f:
            lines = f.readlines()

            start_idx, end_idx = -1, -1
            for i, line in enumerate(lines):
                if line.startswith(f'>>>>>>>start training : {ckpt_path}'):
                    start_idx = i
                    print(f'Found start at line {i}')
                elif (start_idx != -1) and line.startswith('Using GPU'):
                    end_idx = i - 1
                    break
            if end_idx == -1:
                end_idx = len(lines) - 1
            
            res = lines[start_idx:end_idx + 1]

        # find all epoch cost times
        time_lines = []
        for line in res:
            if 'cost time:' in line:
                time_lines.append(float(line.split('cost time:')[1].strip()))
        total_time = sum(time_lines)

        print(f'Total training time for {dataset} with {model}: {total_time:.4f} seconds')

        # find model param
        for line in res:
            if 'model parameter :' in line:
                model_param = int(line.replace('model parameter :', '').strip())
                break
            
        print(f'Model size: {model_param}\n')
        df.append({
            'model': model,
            'dataset': dataset,
            'total_time': total_time,
            'model_param': model_param
        })

import pandas as pd

df = pd.DataFrame(df)

classification_CLS_EigenWorms_DLinear_UEA_ftM_sl17984_ll0_pl0_ma180_iFalse_Exp_0
Found start at line 1288
Total training time for EigenWorms with DLinear: 12.7289 seconds
Model size: 647424005

classification_CLS_InsectWingbeat_DLinear_UEA_ftM_sl22_ll0_pl0_ma7_iFalse_Exp_0
Found start at line 2870
Total training time for InsectWingbeat with DLinear: 1032.4844 seconds
Model size: 45022

classification_CLS_DuckDuckGeese_DLinear_UEA_ftM_sl270_ll0_pl0_ma122_iFalse_Exp_0
Total training time for DuckDuckGeese with DLinear: 0.0000 seconds
Model size: 0

classification_CLS_EigenWorms_LightTS_UEA_ftM_sl17984_ll0_pl0_dm32_cs1799_Exp_0
Found start at line 7829
Total training time for EigenWorms with LightTS: 45.6692 seconds
Model size: 324209097

classification_CLS_InsectWingbeat_LightTS_UEA_ftM_sl22_ll0_pl0_dm128_cs6_Exp_0
Found start at line 8462
Total training time for InsectWingbeat with LightTS: 1461.2879 seconds
Model size: 91936

classification_CLS_DuckDuckGeese_LightTS_UEA_ftM_sl270_ll0_p

In [33]:
df.pivot(index='model', columns='dataset', values='total_time').loc[models]
# df.pivot(index='model', columns='dataset', values='model_param').loc[models]

dataset,DuckDuckGeese,EigenWorms,InsectWingbeat
model,,,
DLinear,0.000000,12.728855,1032.484382
LightTS,18.914346,45.669247,1461.287868
MTSMixer,17.593582,1216.578567,328.329393
TimesNet,62.592612,929.779356,1845.376285
ModernTCN,40.457208,81.146637,34170.504238
TimeMixerPP,261.601660,207.577040,2641.140747
FEDformer,58.950555,366.738265,4903.705489
ETSformer,61.655369,837.680558,2745.528893
Crossformer,116.941233,241.976844,1568.606328


In [21]:
res = '''
Epoch: 1 cost time: 3.1347815990448
Epoch: 1, Steps: 4 | Train Loss: 1.553 Vali Loss: 1.459 Vali Acc: 0.380 Test Loss: 1.459 Test Acc: 0.380
Validation loss decreased (inf --> -0.379985).  Saving model ...
Epoch: 2 cost time: 1.193039894104004
Epoch: 2, Steps: 4 | Train Loss: 1.125 Vali Loss: 1.621 Vali Acc: 0.460 Test Loss: 1.621 Test Acc: 0.460
Validation loss decreased (-0.379985 --> -0.459984).  Saving model ...
Epoch: 3 cost time: 1.7246332168579102
Epoch: 3, Steps: 4 | Train Loss: 0.834 Vali Loss: 1.541 Vali Acc: 0.460 Test Loss: 1.541 Test Acc: 0.460
Validation loss decreased (-0.459984 --> -0.459985).  Saving model ...
Epoch: 4 cost time: 1.511589527130127
Epoch: 4, Steps: 4 | Train Loss: 1.175 Vali Loss: 2.257 Vali Acc: 0.520 Test Loss: 2.257 Test Acc: 0.520
Validation loss decreased (-0.459985 --> -0.519977).  Saving model ...
Epoch: 5 cost time: 1.3083317279815674
Epoch: 5, Steps: 4 | Train Loss: 0.784 Vali Loss: 3.169 Vali Acc: 0.560 Test Loss: 3.169 Test Acc: 0.560
Validation loss decreased (-0.519977 --> -0.559968).  Saving model ...
Epoch: 6 cost time: 1.7685165405273438
Epoch: 6, Steps: 4 | Train Loss: 0.372 Vali Loss: 2.833 Vali Acc: 0.560 Test Loss: 2.833 Test Acc: 0.560
Validation loss decreased (-0.559968 --> -0.559972).  Saving model ...
Epoch: 7 cost time: 1.3013973236083984
Epoch: 7, Steps: 4 | Train Loss: 0.411 Vali Loss: 2.882 Vali Acc: 0.540 Test Loss: 2.882 Test Acc: 0.540
EarlyStopping counter: 1 out of 10
Epoch: 8 cost time: 1.4640600681304932
Epoch: 8, Steps: 4 | Train Loss: 0.239 Vali Loss: 3.317 Vali Acc: 0.600 Test Loss: 3.317 Test Acc: 0.600
Validation loss decreased (-0.559972 --> -0.599967).  Saving model ...
Epoch: 9 cost time: 1.4803683757781982
Epoch: 9, Steps: 4 | Train Loss: 0.769 Vali Loss: 3.776 Vali Acc: 0.540 Test Loss: 3.776 Test Acc: 0.540
EarlyStopping counter: 1 out of 10
Epoch: 10 cost time: 1.5143027305603027
Epoch: 10, Steps: 4 | Train Loss: 0.202 Vali Loss: 4.503 Vali Acc: 0.560 Test Loss: 4.503 Test Acc: 0.560
EarlyStopping counter: 2 out of 10
Epoch: 11 cost time: 1.4936754703521729
Epoch: 11, Steps: 4 | Train Loss: 0.255 Vali Loss: 6.609 Vali Acc: 0.620 Test Loss: 6.609 Test Acc: 0.620
Validation loss decreased (-0.599967 --> -0.619934).  Saving model ...
Epoch: 12 cost time: 1.4309144020080566
Epoch: 12, Steps: 4 | Train Loss: 0.275 Vali Loss: 5.025 Vali Acc: 0.540 Test Loss: 5.025 Test Acc: 0.540
EarlyStopping counter: 1 out of 10
Epoch: 13 cost time: 1.5157601833343506
Epoch: 13, Steps: 4 | Train Loss: 0.085 Vali Loss: 5.070 Vali Acc: 0.580 Test Loss: 5.070 Test Acc: 0.580
EarlyStopping counter: 2 out of 10
Epoch: 14 cost time: 1.5339858531951904
Epoch: 14, Steps: 4 | Train Loss: 0.341 Vali Loss: 5.502 Vali Acc: 0.620 Test Loss: 5.502 Test Acc: 0.620
Validation loss decreased (-0.619934 --> -0.619945).  Saving model ...
Epoch: 15 cost time: 1.4463605880737305
Epoch: 15, Steps: 4 | Train Loss: 0.204 Vali Loss: 6.499 Vali Acc: 0.620 Test Loss: 6.499 Test Acc: 0.620
EarlyStopping counter: 1 out of 10
Epoch: 16 cost time: 1.6992385387420654
Epoch: 16, Steps: 4 | Train Loss: 0.124 Vali Loss: 5.802 Vali Acc: 0.540 Test Loss: 5.802 Test Acc: 0.540
EarlyStopping counter: 2 out of 10
Epoch: 17 cost time: 1.365335464477539
Epoch: 17, Steps: 4 | Train Loss: 0.100 Vali Loss: 6.757 Vali Acc: 0.520 Test Loss: 6.757 Test Acc: 0.520
EarlyStopping counter: 3 out of 10
Epoch: 18 cost time: 1.600268840789795
Epoch: 18, Steps: 4 | Train Loss: 0.051 Vali Loss: 6.582 Vali Acc: 0.560 Test Loss: 6.582 Test Acc: 0.560
EarlyStopping counter: 4 out of 10
Epoch: 19 cost time: 1.5903191566467285
Epoch: 19, Steps: 4 | Train Loss: 0.422 Vali Loss: 6.877 Vali Acc: 0.660 Test Loss: 6.877 Test Acc: 0.660
Validation loss decreased (-0.619945 --> -0.659931).  Saving model ...
Epoch: 20 cost time: 1.7712056636810303
Epoch: 20, Steps: 4 | Train Loss: 0.046 Vali Loss: 6.973 Vali Acc: 0.620 Test Loss: 6.973 Test Acc: 0.620
EarlyStopping counter: 1 out of 10
Epoch: 21 cost time: 1.7261974811553955
Epoch: 21, Steps: 4 | Train Loss: 0.476 Vali Loss: 7.728 Vali Acc: 0.540 Test Loss: 7.728 Test Acc: 0.540
EarlyStopping counter: 2 out of 10
Epoch: 22 cost time: 1.5567142963409424
Epoch: 22, Steps: 4 | Train Loss: 0.008 Vali Loss: 7.721 Vali Acc: 0.600 Test Loss: 7.721 Test Acc: 0.600
EarlyStopping counter: 3 out of 10
Epoch: 23 cost time: 1.5248382091522217
Epoch: 23, Steps: 4 | Train Loss: 0.001 Vali Loss: 7.677 Vali Acc: 0.600 Test Loss: 7.677 Test Acc: 0.600
EarlyStopping counter: 4 out of 10
Epoch: 24 cost time: 1.5881316661834717
Epoch: 24, Steps: 4 | Train Loss: 0.001 Vali Loss: 8.251 Vali Acc: 0.620 Test Loss: 8.251 Test Acc: 0.620
EarlyStopping counter: 5 out of 10
Epoch: 25 cost time: 1.5788254737854004
Epoch: 25, Steps: 4 | Train Loss: 0.001 Vali Loss: 8.493 Vali Acc: 0.580 Test Loss: 8.493 Test Acc: 0.580
EarlyStopping counter: 6 out of 10
Epoch: 26 cost time: 2.0245749950408936
Epoch: 26, Steps: 4 | Train Loss: 0.001 Vali Loss: 8.506 Vali Acc: 0.580 Test Loss: 8.506 Test Acc: 0.580
EarlyStopping counter: 7 out of 10
Epoch: 27 cost time: 1.4949703216552734
Epoch: 27, Steps: 4 | Train Loss: 0.000 Vali Loss: 8.711 Vali Acc: 0.560 Test Loss: 8.711 Test Acc: 0.560
EarlyStopping counter: 8 out of 10
Epoch: 28 cost time: 1.9367666244506836
Epoch: 28, Steps: 4 | Train Loss: 0.000 Vali Loss: 8.961 Vali Acc: 0.560 Test Loss: 8.961 Test Acc: 0.560
EarlyStopping counter: 9 out of 10
Epoch: 29 cost time: 1.6838936805725098
Epoch: 29, Steps: 4 | Train Loss: 0.000 Vali Loss: 9.166 Vali Acc: 0.560 Test Loss: 9.166 Test Acc: 0.560
EarlyStopping counter: 10 out of 10
Early stopping
>>>>>>>testing : classification_CLS_DuckDuckGeese_DLinear_UEA_ftM_sl270_ll0_pl0_ma122_Exp_0<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
test shape: torch.Size([50, 5]) torch.Size([50, 1])
accuracy:0.66
model parameter : 1962095
model size : 7.49MB
'''.split('\n')

# find all epoch cost times
time_lines = []
for line in res:
    if 'cost time:' in line:
        time_lines.append(float(line.split('cost time:')[1].strip()))
total_time = sum(time_lines)

print(f'Total training time for {dataset} with {model}: {total_time:.4f} seconds')

# find model param
for line in res:
    if 'model parameter :' in line:
        model_param = int(line.replace('model parameter :', '').strip())
        break
    
print(f'Model size: {model_param}\n')

Total training time for DuckDuckGeese with MambaSL: 46.9630 seconds
Model size: 1962095

